# Built-in algorithm: Random Cut Forest

**Module 3, slide 13** - the deck's optional *Built-in algorithm* demo.

The slide asks for four things: launch a notebook, import it in Studio, set up the
libraries for a built-in algorithm, and train with Random Cut Forest. Its own
speaker notes say **stop after training** - do not continue to inference.

**The one-sentence pitch:** every other model in this series is supervised - we
invented a `high_demand` label and asked a model to predict it. This one gets *no
label at all*.

**Cost:** ml.m5.large for ~3 minutes. Cents. No endpoint, so nothing to tear down.


## 1. Setup

Nothing RCF-specific yet - this is just "where am I and what can I use".

`get_execution_role()` works inside Studio because the notebook already runs as the
domain's execution role. The bucket is discovered rather than typed, so there is
nothing to edit live.


In [ ]:
import boto3

# The Studio image ships SageMaker SDK **v3**; a laptop install is usually v2.
# They disagree about where things live, and v3 does not just warn -- importing a
# removed module raises ModuleNotFoundError, which kills the cell outright.
#   v2:  from sagemaker import image_uris
#   v3:  from sagemaker.core import image_uris
# Try v3 first, fall back to v2. This is the only line in the notebook that
# touches the SDK at all.
try:
    from sagemaker.core import image_uris          # SDK v3
    _SDK = 'v3'
except ImportError:
    from sagemaker import image_uris               # SDK v2
    _SDK = 'v2'

sess    = boto3.Session()
region  = sess.region_name or 'us-east-1'
ident   = sess.client('sts').get_caller_identity()
account = ident['Account']
sm      = sess.client('sagemaker')
s3      = sess.client('s3')

# Derive the execution role from STS instead of sagemaker.get_execution_role().
# `sagemaker.session` was REMOVED in v3 and the migration guide does not document
# a replacement, so the SDK is simply not a safe place to get this. In Studio the
# caller is already the execution role:
#   arn:aws:sts::<acct>:assumed-role/<RoleName>/<session>
_arn = ident['Arn']
if ':assumed-role/' in _arn:
    ROLE = f"arn:aws:iam::{account}:role/{_arn.split(':assumed-role/')[1].split('/')[0]}"
else:
    ROLE = f'arn:aws:iam::{account}:role/roi-smdemo-exec-role'

buckets = [b['Name'] for b in s3.list_buckets()['Buckets'] if 'smdemo' in b['Name']]
BUCKET  = buckets[0] if buckets else f'roi-smdemo-{account}-{region}'

print(f'sdk    {_SDK}')
print(f'region {region}  account {account}')
print(f'bucket {BUCKET}')
print(f'role   {ROLE}')


## 2. The data - and the column we delete

This is the **same feature matrix** the supervised demos train on. That is the
point: the room has already seen this table.

RCF is unsupervised, so the label has to go. Leaving `high_demand` in would be
target leakage of the most literal kind - the model would be handed the answer it is
supposed to be ignorant of, and the column it found most useful for isolating points
would be the label itself.


In [ ]:
SOURCE = 'ml/gas/autopilot/train.csv'

raw    = s3.get_object(Bucket=BUCKET, Key=SOURCE)['Body'].read().decode()
lines  = raw.strip().split(chr(10))
header = lines[0].split(',')

print(f'{len(lines)-1:,} rows, {len(header)} columns')
print()
print('LABEL WE DROP :', header[0])
print('FEATURES KEPT :', len(header)-1)
for c in header[1:]:
    print('   ', c)


### Write it back out headerless

Built-in algorithms take **headerless CSV**. The container cannot see a schema -
which is exactly why `feature_dim` has to be a hyperparameter further down.


In [ ]:
import io

label_idx = header.index('high_demand')
out = io.StringIO()
for row in lines[1:]:
    cells = row.split(',')
    out.write(','.join(c for i, c in enumerate(cells) if i != label_idx) + chr(10))

PREFIX = 'ml/gas/rcf'
s3.put_object(Bucket=BUCKET, Key=f'{PREFIX}/train/train.csv',
              Body=out.getvalue().encode())

FEATURE_DIM = len(header) - 1
print(f'wrote s3://{BUCKET}/{PREFIX}/train/train.csv')
print(f'feature_dim = {FEATURE_DIM}   <-- computed, never typed by hand')


## 3. "Set up the libraries to use a built-in algorithm"

This is the slide's third bullet, and it is **one line**.

There is no training script. No `train.py`, no `requirements.txt`, no model code at
all. A built-in algorithm is an AWS-maintained **container image** you address by
name. You bring data and hyperparameters; that is the whole contract.

> Say this out loud: *"I wrote no model code for this one."*


In [ ]:
image = image_uris.retrieve('randomcutforest', region)
print(image)


## 4. Train

**This is the cooking-show moment.** Run the cell, show the job go `InProgress`,
then stop narrating - it takes about three minutes - and jump to section 5, where
one is already finished.

Three things worth pointing at in this call:

| | |
|---|---|
| `num_trees=50` | how many random trees vote on each point |
| `num_samples_per_tree=256` | how much data each tree sees |
| `label_size=0` | how you tell a built-in there is **no answer column** |

Omit `label_size=0` and RCF silently treats your first feature as a label.


In [ ]:
import time
job = 'roi-smdemo-gas-rcf-' + time.strftime('%y%m%d-%H%M')

sm.create_training_job(
    TrainingJobName=job,
    AlgorithmSpecification={'TrainingImage': image, 'TrainingInputMode': 'File'},
    RoleArn=ROLE,
    HyperParameters={
        'num_trees': '50',
        'num_samples_per_tree': '256',
        'feature_dim': str(FEATURE_DIM),   # mismatch here = job dies at startup
    },
    InputDataConfig=[{
        'ChannelName': 'train',
        'DataSource': {'S3DataSource': {
            'S3DataType': 'S3Prefix',
            'S3Uri': f's3://{BUCKET}/{PREFIX}/train/',
            'S3DataDistributionType': 'ShardedByS3Key',
        }},
        'ContentType': 'text/csv;label_size=0',
        'CompressionType': 'None',
    }],
    OutputDataConfig={'S3OutputPath': f's3://{BUCKET}/{PREFIX}/output/'},
    ResourceConfig={'InstanceType': 'ml.m5.large', 'InstanceCount': 1,
                    'VolumeSizeInGB': 10},
    StoppingCondition={'MaxRuntimeInSeconds': 900},
)
print(job, '->', sm.describe_training_job(TrainingJobName=job)['TrainingJobStatus'])
print('~3 minutes. Move to section 5 and come back.')


## 5. "Here's one I made earlier"

Pulls the most recent **Completed** RCF job, so this works whether or not the cell
above has finished.

> Note: no `MaxResults` alongside `NameContains`. SageMaker applies the limit to the
> page it fetches *before* the name filter, so `MaxResults=1` silently returns
> nothing as soon as another job is newer. That bug bit several scripts in this repo.


In [ ]:
done = sm.list_training_jobs(NameContains='roi-smdemo-gas-rcf',
                             StatusEquals='Completed',
                             SortBy='CreationTime',
                             SortOrder='Descending')['TrainingJobSummaries']
d = sm.describe_training_job(TrainingJobName=done[0]['TrainingJobName'])

print(d['TrainingJobName'], d['TrainingJobStatus'])
print('hyperparameters :', d['HyperParameters'])
print('instance        :', d['ResourceConfig']['InstanceType'])
print('billable seconds:', d['BillableTimeInSeconds'])
print('model           :', d['ModelArtifacts']['S3ModelArtifacts'])
print()
print('METRICS:')
for m in d.get('FinalMetricDataList', []):
    print(f"   {m['MetricName']:20} {m['Value']:,.1f}")


### The punchline - look at what is *missing*

`train:progress` and `train:throughput`. That is all.

**There is no accuracy. No AUC. No objective metric at all** - and there cannot be,
because there is no ground truth to score against. That is what unsupervised means.

> *"Every other job today ends with a number saying how good it is. This one can't
> have one. So 'is this model good?' has no answer here. The real question is whether
> the things it flags turn out to be worth a truck roll - and only your field crews
> can answer that."*

Which sets up the next demo, where you **do** get a number - and the number lies.

---

## Stop here

The slide says end after training. The second reason: scoring points means standing
up an endpoint, and endpoints bill by the hour. Nothing in this notebook created
one, so there is nothing to tear down.
